# 05 Prior Authorization Risk Scoring Demo

Purpose: create a CMS-aligned PA risk prioritization prototype. Public CMS PA artifacts provide reporting schema and policy timing rules, not mature request-level labels. Therefore this notebook uses a seeded demo dataset plus transparent rule features.

In [ ]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

## Run PA Demo Model

Why: the model demonstrates feature engineering and classifier workflow while staying honest about public-data limits.

In [ ]:
from src.models.pa_classifier import run_pa_model

pa_predictions, pa_metrics = run_pa_model()
display(pa_metrics)
display(pa_predictions.sort_values("hybrid_denial_risk", ascending=False).head(20))

## PA Risk Visuals

Elements: histograms show risk distribution; bars show high-risk groups by procedure/payer.

In [ ]:
fig = px.histogram(pa_predictions, x="hybrid_denial_risk", color="hybrid_risk_bucket", title="PA Hybrid Denial Risk Distribution")
fig.show()

risk_by_proc = (
    pa_predictions.groupby(["procedure_type", "payer_type"], as_index=False)
    .agg(avg_hybrid_denial_risk=("hybrid_denial_risk", "mean"), high_risk_cases=("hybrid_risk_bucket", lambda s: int((s == "high").sum())))
    .sort_values("avg_hybrid_denial_risk", ascending=False)
)
risk_by_proc.to_csv(TABLE_DIR / "pa_risk_by_procedure_payer.csv", index=False)
fig = px.bar(risk_by_proc, x="procedure_type", y="avg_hybrid_denial_risk", color="payer_type", title="Average PA Denial Risk by Procedure and Payer")
fig.show()
display(risk_by_proc)

## PA Decision

The model accuracy is useful for demonstration, but the correct business framing is risk prioritization. The output tells a team which authorization requests need stronger documentation before submission; it does not claim payer-specific production denial prediction.